# S10 — From LeNet to ResNet

**Week 6 · Mon Sep 28, 2026 · Module 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s10_from_lenet_to_resnet.ipynb)

Every cell below is a worked example from the [S10 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s10/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s10.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s10.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## The architecture zoo: depth is not parameters


*Expected output starts with:* `conv layers (13):         14,714,688 params`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# VGG-16, built from the paper's configuration table (Simonyan & Zisserman,
# ICLR 2015, configuration D): numbers are output channels, 'M' is 2x2 max pool.
cfg = [64, 64, "M", 128, 128, "M", 256, 256, 256, "M",
       512, 512, 512, "M", 512, 512, 512, "M"]

features, c_in = [], 3
for v in cfg:
    if v == "M":
        features.append(nn.MaxPool2d(2))
    else:
        features += [nn.Conv2d(c_in, v, 3, padding=1), nn.ReLU()]
        c_in = v
features = nn.Sequential(*features)

classifier = nn.Sequential(               # input image 224x224 -> 512 x 7 x 7
    nn.Flatten(),
    nn.Linear(512 * 7 * 7, 4096), nn.ReLU(), nn.Dropout(),
    nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(),
    nn.Linear(4096, 1000),
)

def n_params(m):
    return sum(p.numel() for p in m.parameters())

conv_p, fc_p = n_params(features), n_params(classifier)
print(f"conv layers (13):       {conv_p:12,d} params")
print(f"fully connected (3):    {fc_p:12,d} params")
print(f"total:                  {conv_p + fc_p:12,d} params")
print(f"share held by the FC head: {fc_p / (conv_p + fc_p):.1%}")
print(f"first FC layer alone:   {512 * 7 * 7 * 4096 + 4096:12,d} params")

## Why depth was hard: the vanishing gradient, measured


*Expected output starts with:* `depth  2: ||grad|| at output layer = 4.18e-01, at first layer = 1.68e-02, ratio = 4.0e-0`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def make_plain_net(depth, width=32):
    """A deep stack of Linear + Sigmoid layers (early-1990s style)."""
    layers = []
    for _ in range(depth):
        layers += [nn.Linear(width, width), nn.Sigmoid()]
    layers += [nn.Linear(width, 1)]
    return nn.Sequential(*layers)

x = torch.randn(64, 32)
y = torch.randn(64, 1)

for depth in [2, 6, 12, 24, 48]:
    torch.manual_seed(0)
    net = make_plain_net(depth)
    loss = nn.functional.mse_loss(net(x), y)
    loss.backward()
    first = net[0]   # the layer closest to the input
    last = net[-1]   # the output layer
    g_first = first.weight.grad.norm().item()
    g_last = last.weight.grad.norm().item()
    print(f"depth {depth:2d}: ||grad|| at output layer = {g_last:.2e}, "
          f"at first layer = {g_first:.2e}, ratio = {g_first / g_last:.1e}")

## Batch normalization: what it actually buys


*Expected output starts with:* `without BN: act std at block 1/10/20/30 = 1.7e-01/4.5e-02/4.1e-02/4.1e-02, first-layer g`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def make_stack(depth, use_bn, scale, width=64):
    layers = []
    for _ in range(depth):
        layers.append(nn.Linear(width, width))
        if use_bn:
            layers.append(nn.BatchNorm1d(width))
        layers.append(nn.ReLU())
    net = nn.Sequential(*layers)
    with torch.no_grad():
        for p in net.parameters():
            if p.dim() == 2:            # weight matrices only
                p.mul_(scale)           # deliberately under-scaled init
    return net

def probe(depth, use_bn, scale=0.5):
    torch.manual_seed(0)
    net = make_stack(depth, use_bn, scale)
    x = torch.randn(256, 64)
    # Record activation std after selected layers.
    h, stds = x, {}
    per_block = 3 if use_bn else 2
    for i, layer in enumerate(net):
        h = layer(h)
        block = i // per_block + 1
        if isinstance(layer, nn.ReLU) and block in (1, 10, 20, 30):
            stds[block] = h.std().item()
    loss = net(x).pow(2).mean()
    loss.backward()
    g1 = net[0].weight.grad.norm().item()
    tag = "with BN   " if use_bn else "without BN"
    print(f"{tag}: act std at block 1/10/20/30 = "
          + "/".join(f"{stds[b]:.1e}" for b in (1, 10, 20, 30))
          + f", first-layer grad norm = {g1:.1e}")

probe(30, use_bn=False)
probe(30, use_bn=True)

## The degradation experiment, reproduced in miniature


*Expected output starts with:* `blocks  plain train acc  residual train acc`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
torch.set_num_threads(1)

# The degradation experiment in miniature: does adding depth make the
# TRAINING fit worse? Plain vs residual blocks, four depths, same budget.

class Block(nn.Module):
    def __init__(self, width, skip):
        super().__init__()
        self.fc1 = nn.Linear(width, width)
        self.fc2 = nn.Linear(width, width)
        self.relu = nn.ReLU()
        self.skip = skip

    def forward(self, x):
        out = self.fc2(self.relu(self.fc1(x)))
        if self.skip:
            out = out + x
        return self.relu(out)

def make_data(n, rng):
    X = rng.normal(0.0, 0.5, size=(n, 2)).astype(np.float32)
    y = (X[:, 0] * X[:, 1] > 0).astype(np.int64)     # XOR-quadrant task
    X += rng.normal(0.0, 0.1, size=X.shape).astype(np.float32)
    return torch.from_numpy(X), torch.from_numpy(y)

def train_accuracy(n_blocks, skip, seed, width=32, steps=600):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    X, y = make_data(256, rng)
    net = nn.Sequential(nn.Linear(2, width), nn.ReLU(),
                        *[Block(width, skip) for _ in range(n_blocks)],
                        nn.Linear(width, 2))
    opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(steps):
        idx = torch.randint(0, len(X), (64,))
        opt.zero_grad()
        loss_fn(net(X[idx]), y[idx]).backward()
        opt.step()
    with torch.no_grad():
        return (net(X).argmax(1) == y).float().mean().item()

print(f"{'blocks':>6} {'plain train acc':>16} {'residual train acc':>19}")
for n in [1, 4, 16, 32]:
    row = []
    for skip in [False, True]:
        row.append(np.mean([train_accuracy(n, skip, seed) for seed in range(3)]))
    print(f"{n:6d} {row[0]:16.4f} {row[1]:19.4f}")

## ResNet: learn the residual


*Expected output starts with:* `blocks        plain     residual`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

class Block(nn.Module):
    """Two 3x3 convs. With skip=True this is a basic ResNet block."""
    def __init__(self, channels, skip):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.relu = nn.ReLU()
        self.skip = skip

    def forward(self, x):
        out = self.conv2(self.relu(self.conv1(x)))
        if self.skip:
            out = out + x          # the residual connection
        return self.relu(out)

def grad_at_first_layer(n_blocks, skip, scale):
    torch.manual_seed(0)
    blocks = nn.Sequential(*[Block(8, skip) for _ in range(n_blocks)])
    # Shrink the conv weights so each block is a contraction, as
    # under-scaled initializations were before He init (2015).
    with torch.no_grad():
        for p in blocks.parameters():
            if p.dim() == 4:
                p.mul_(scale)
    x = torch.randn(4, 8, 8, 8)
    out = blocks(x)
    loss = out.pow(2).mean()
    loss.backward()
    return blocks[0].conv1.weight.grad.norm().item()

print(f"{'blocks':>6} {'plain':>12} {'residual':>12}")
for n in [2, 8, 16, 32]:
    g_plain = grad_at_first_layer(n, skip=False, scale=0.5)
    g_res = grad_at_first_layer(n, skip=True, scale=0.5)
    print(f"{n:6d} {g_plain:12.2e} {g_res:12.2e}")

## Try it yourself

1. Rerun the vanishing-gradient demo with `nn.ReLU()` in place of `nn.Sigmoid()`. Measure how the first-layer gradient norm scales with depth now, and reconcile what you see with the claim that ReLU "solves" vanishing gradients.
2. In the residual example, vary `scale` over 0.4, 0.5, 0.7, and 1.0 for both plain and residual stacks at 16 blocks. Which configuration is most sensitive to initialization scale, and in which direction can gradients *explode*?
3. Add a BatchNorm layer after each conv in `Block` (the real ResNet ordering is conv-BN-ReLU) and repeat the comparison at `scale=0.5`. Does normalization alone rescue the plain stack? Does it change the residual stack?
4. Build the smallest network you can that gets 100% training accuracy on the "+" vs "x" task from [Section 2.3]({{ '/readings/ch2/transfer-learning/' | relative_url }})'s first example, with and without skips. At what depth does the plain version start losing to the residual one?


---

Full discussion of everything above: [S10 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s10/).
